In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter

# ==========================================================
# PREPARACIÓN DE LOS DATOS
# ==========================================================

# Normalizar género
df_municipio_genero["genero_limpio"] = (
    df_municipio_genero["genero"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Orden de los tipos de delito
orden_delitos = [
    "Amenazas",
    "Delitos sexuales",
    "Homicidio",
    "Hurto a residencias y entidades comerciales",
    "Hurto de motocicletas y automotores",
    "Lesiones personales",
    "Violencia intrafamiliar"
]

# ----------------------------------------------------------
# Total de registros por municipio y tipo de delito
# Incluye FEMENINO, MASCULINO y NO REPORTA
# ----------------------------------------------------------

totales_municipio_delito = (
    df_municipio_genero
    .groupby(
        ["municipio", "departamento", "tipo_delito"],
        as_index=False
    )["cantidad_delitos"]
    .sum()
    .rename(columns={"cantidad_delitos": "total_delito"})
)

# ----------------------------------------------------------
# Orden de municipios según su volumen total
# ----------------------------------------------------------

orden_municipios = (
    df_municipio_genero
    .groupby(["municipio", "departamento"])["cantidad_delitos"]
    .sum()
    .sort_values(ascending=False)
    .index
)

# ==========================================================
# FUNCIÓN PARA GENERAR EL HEATMAP
# ==========================================================

def crear_heatmap_genero_porcentaje(df, genero, titulo):

    # ------------------------------------------------------
    # Filtrar el género
    # ------------------------------------------------------

    df_genero = df[
        df["genero_limpio"] == genero
    ].copy()

    if df_genero.empty:
        print(f"No se encontraron registros para {genero}.")
        return

    # ------------------------------------------------------
    # Unir el total municipio × delito
    # ------------------------------------------------------

    df_genero = df_genero.merge(
        totales_municipio_delito,
        on=["municipio", "departamento", "tipo_delito"],
        how="left"
    )

    # ------------------------------------------------------
    # Calcular porcentaje
    # ------------------------------------------------------

    df_genero["porcentaje"] = (
        df_genero["cantidad_delitos"]
        / df_genero["total_delito"]
    ) * 100

    # ------------------------------------------------------
    # Matriz de cantidades
    # ------------------------------------------------------

    tabla_cantidad = df_genero.pivot_table(
        index=["municipio", "departamento"],
        columns="tipo_delito",
        values="cantidad_delitos",
        aggfunc="sum",
        fill_value=0
    )

    # ------------------------------------------------------
    # Matriz de porcentajes
    # ------------------------------------------------------

    tabla_porcentaje = df_genero.pivot_table(
        index=["municipio", "departamento"],
        columns="tipo_delito",
        values="porcentaje",
        aggfunc="sum",
        fill_value=0
    )

    # Asegurar orden de columnas
    tabla_cantidad = tabla_cantidad.reindex(
        columns=orden_delitos,
        fill_value=0
    )

    tabla_porcentaje = tabla_porcentaje.reindex(
        columns=orden_delitos,
        fill_value=0
    )

    # Asegurar orden de municipios
    tabla_cantidad = tabla_cantidad.reindex(
        orden_municipios,
        fill_value=0
    )

    tabla_porcentaje = tabla_porcentaje.reindex(
        orden_municipios,
        fill_value=0
    )

    datos = tabla_cantidad.to_numpy()
    porcentajes = tabla_porcentaje.to_numpy()

    # ------------------------------------------------------
    # Escala logarítmica para el color
    # ------------------------------------------------------

    valores_positivos = datos[datos > 0]

    if valores_positivos.size == 0:
        print(f"No hay valores positivos para {genero}.")
        return

    norm = LogNorm(
        vmin=valores_positivos.min(),
        vmax=valores_positivos.max()
    )

    # ======================================================
    # FIGURA
    # ======================================================

    fig, ax = plt.subplots(figsize=(15, 8))

    imagen = ax.imshow(
        datos,
        aspect="auto",
        norm=norm
    )

    # ------------------------------------------------------
    # Eje X
    # ------------------------------------------------------

    etiquetas_delitos = [
        "Amenazas",
        "Delitos\nsexuales",
        "Homicidio",
        "Hurto a residencias\ny entidades \ncomerciales",
        "Hurto de \nmotocicletas y \nautomotores",
        "Lesiones\npersonales",
        "Violencia\nintrafamiliar"
    ]

    ax.set_xticks(np.arange(len(orden_delitos)))
    ax.set_xticklabels(
        etiquetas_delitos,
        rotation=0,
        ha="center"
    )

    ax.set_xlabel(
        "Tipo de delito",
        labelpad=20
    )

    # ------------------------------------------------------
    # Eje Y
    # ------------------------------------------------------

    etiquetas_municipios = [
        f"{municipio} — {departamento}"
        for municipio, departamento in orden_municipios
    ]

    ax.set_yticks(np.arange(len(etiquetas_municipios)))
    ax.set_yticklabels(etiquetas_municipios)

    ax.set_ylabel("Municipio")

    ax.set_title(titulo)

    # ------------------------------------------------------
    # Etiquetas: cantidad + porcentaje
    # ------------------------------------------------------

    for i in range(datos.shape[0]):
        for j in range(datos.shape[1]):

            cantidad = datos[i, j]
            porcentaje = porcentajes[i, j]

            if cantidad > 0:

                texto_cantidad = (
                    f"{int(cantidad):,}".replace(",", ".")
                )

                texto = (
                    f"{texto_cantidad}\n"
                    f"({porcentaje:.1f} %)"
                )
                # Obtener el color de fondo de la celda
                rgba = imagen.cmap(norm(cantidad))

                # Calcular luminancia del color
                r, g, b, _ = rgba

                luminancia = (
                    0.299 * r +
                    0.587 * g +
                    0.114 * b
                )

                # Fondo oscuro -> texto blanco
                # Fondo claro -> texto negro
                color_texto = "white" if luminancia < 0.5 else "black"

                ax.text(
                    j,
                    i,
                    texto,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=color_texto,
                    linespacing=1.1
                )

    # ------------------------------------------------------
    # Barra de color
    # ------------------------------------------------------

    cbar = fig.colorbar(
        imagen,
        ax=ax
    )

    cbar.set_label(
        "Cantidad de registros"
    )

    cbar.ax.yaxis.set_major_formatter(
        FuncFormatter(
            lambda x, pos:
            f"{int(x):,}".replace(",", ".")
        )
    )

    plt.tight_layout()
    plt.show()

In [ ]:
crear_heatmap_genero_porcentaje(
    df_municipio_genero,
    "FEMENINO",
    "Distribución de los registros asociados al género femenino por municipio y tipo de delito, 2014–2024"
)

crear_heatmap_genero_porcentaje(
    df_municipio_genero,
    "MASCULINO",
    "Distribución de los registros asociados al género masculino por municipio y tipo de delito, 2014–2024"
)

tipo delito - municipio - arma

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import matplotlib.colors as mcolors

# ==========================================================
# 1. DATOS
# ==========================================================

df = df_armas_municipios.copy()

df["categoria_grafico"] = df["categoria_grafico"].replace(
    "ARMA BLANCA / CORTOPUNZANTE",
    "ARMA BLANCA"
)
df["categoria_grafico"] = (
    df["categoria_grafico"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.capitalize()
)

# ==========================================================
# 2. LIMPIEZA DE TEXTO
# ==========================================================

for columna in ["municipio", "tipo_delito", "categoria_grafico"]:
    df[columna] = (
        df[columna]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

df["municipio"] = df["municipio"].replace({
    "ApartadÃ³": "Apartadó",
    "MedellÃ­n (Ct)": "Medellín (Ct)",
    "Itagui": "Itagüí"
})

# ==========================================================
# 3. ORDEN DE LOS MUNICIPIOS
# ==========================================================

orden_municipios = [
    "Apartadó",
    "Bello",
    "Caucasia",
    "Copacabana",
    "Envigado",
    "Itagüí",
    "Medellín (Ct)",
    "Rionegro",
    "Sabaneta",
    "Turbo"
]

df["municipio"] = pd.Categorical(
    df["municipio"],
    categories=orden_municipios,
    ordered=True
)

# ==========================================================
# 4. ORDEN DE LOS DELITOS + LETRAS DE SUBFIGURA
# ==========================================================

orden_delitos = [
    "Amenazas",
    "Delitos sexuales",
    "Homicidio",
    "Hurto a residencias y entidades comerciales",
    "Hurto de motocicletas y automotores",
    "Lesiones personales",
    "Violencia intrafamiliar"
]

letras_figura = {
    "Amenazas": "a",
    "Delitos sexuales": "b",
    "Homicidio": "c",
    "Hurto a residencias y entidades comerciales": "d",
    "Hurto de motocicletas y automotores": "e",
    "Lesiones personales": "f",
    "Violencia intrafamiliar": "g"
}

# ==========================================================
# 5. ORDEN DE LAS CATEGORÍAS
# ==========================================================

orden_categorias = [
    "Sin empleo de armas",
    "Arma de fuego",
    "Arma blanca",
    "Contundentes",
    "Palancas",
    "Llave maestra",
    "Vehículo",
    "Moto",
    "Ácido",
    "No reportado",
    "Otros"
]

# ==========================================================
# 6. PALETA DE COLORES
# ==========================================================

colores_base = [
    "#1f77b4",  # azul
    "#ff7f0e",  # naranja
    "#2ca02c",  # verde
    "#d62728",  # rojo
    "#9467bd",  # morado
    "#8c564b",  # café
    "#e377c2",  # rosa
    "#7f7f7f",  # gris
    "#bcbd22",  # oliva
    "#17becf",  # cian
    "#aec7e8"   # azul claro
]

mapa_colores = {
    categoria: colores_base[i]
    for i, categoria in enumerate(orden_categorias)
}

# ==========================================================
# 7. NOMBRES DE LOS MUNICIPIOS PARA EL EJE X
# ==========================================================

nombres_x = [
    "Apartadó",
    "Bello",
    "Caucasia",
    "Copacabana",
    "Envigado",
    "Itagüí",
    "Medellín\n(Ct)",
    "Rionegro",
    "Sabaneta",
    "Turbo"
]

# ==========================================================
# 8. NOMBRES DE LOS DELITOS
# ==========================================================

titulos_delitos = {
    "Amenazas": "Amenazas",
    "Delitos sexuales": "Delitos sexuales",
    "Homicidio": "Homicidio",
    "Hurto a residencias y entidades comerciales": "Hurto a residencias y entidades comerciales",
    "Hurto de motocicletas y automotores": "Hurto de motocicletas y automotores",
    "Lesiones personales": "Lesiones personales",
    "Violencia intrafamiliar": "Violencia intrafamiliar"
}

# ==========================================================
# 9. GENERAR LAS 7 GRÁFICAS (Figura 4-7.a a 4-7.g)
# ==========================================================

for delito in orden_delitos:

    letra = letras_figura[delito]

    datos = df[df["tipo_delito"] == delito].copy()

    tabla = datos.pivot_table(
        index="municipio",
        columns="categoria_grafico",
        values="cantidad_registros",
        aggfunc="sum",
        fill_value=0
    )

    tabla = tabla.reindex(orden_municipios, fill_value=0)

    categorias_presentes = [
        categoria
        for categoria in orden_categorias
        if categoria in tabla.columns
    ]

    tabla_porcentaje = (
        tabla[categorias_presentes]
        .div(tabla[categorias_presentes].sum(axis=1), axis=0)
        * 100
    )

    # ======================================================
    # GRÁFICO
    # ======================================================

    fig, ax = plt.subplots(figsize=(12, 5.2))

    bottom = np.zeros(len(tabla_porcentaje))

    for categoria in categorias_presentes:

        valores = tabla_porcentaje[categoria].values

        barras = ax.bar(
            range(len(tabla_porcentaje)),
            valores,
            bottom=bottom,
            width=0.72,
            label=categoria,
            color=mapa_colores.get(categoria, "#999999"),
            edgecolor="white",
            linewidth=0.4
        )

        color_barra = mapa_colores.get(categoria, "#999999")
        r, g, b = mcolors.to_rgb(color_barra)
        luminosidad = 0.299 * r + 0.587 * g + 0.114 * b
        color_texto = "white" if luminosidad < 0.55 else "black"

        for j, (bar, val) in enumerate(zip(barras, valores)):
            if val >= 5:
                altura = bottom[j] + val / 2
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    altura,
                    f"{val:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=8.5,
                    color=color_texto,
                    fontweight="medium"
                )

        bottom += valores

    # ======================================================
    # FORMATO
    # ======================================================

    ax.set_title(
        f"Figura 4-7.{letra}. {titulos_delitos[delito]}",
        fontsize=11,
        pad=14
    )

    ax.set_xlabel("Municipio", fontsize=10, labelpad=5)
    ax.set_ylabel("Porcentaje de registros", fontsize=10)

    ax.set_xticks(range(len(tabla_porcentaje)))
    ax.set_xticklabels(nombres_x, rotation=0, ha="center", fontsize=9)

    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(PercentFormatter(100))
    ax.tick_params(axis="y", labelsize=9)

    ax.grid(axis="y", alpha=0.35)
    ax.set_axisbelow(True)

    ax.legend(
        title="Arma o medio empleado",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        fontsize=9,
        title_fontsize=10
    )

    plt.tight_layout()
    plt.show()
    plt.close()

#### **Analisis tiempo municipio**

In [ ]:
import matplotlib.pyplot as plt

# ==========================================================
# PREPARACIÓN DE DATOS
# ==========================================================

df_plot = df_municipios_temporal.copy()

df_plot["anio"] = df_plot["anio"].astype(int)
df_plot["total_afectados"] = df_plot["total_afectados"].astype(float)

# ==========================================================
# IDENTIFICAR MUNICIPIOS
# ==========================================================

municipios = df_plot["municipio"].dropna().unique()

# Identificar Medellín directamente desde el DataFrame
municipio_medellin = [
    m for m in municipios
    if "Medell" in str(m)
][0]

# Los otros municipios
municipios_otros = sorted([
    m for m in municipios
    if m != municipio_medellin
])


# ==========================================================
# CREAR FIGURA
# ==========================================================

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(13, 9),
    sharex=True,
    gridspec_kw={
        "height_ratios": [1, 1],
        "hspace": 0.25
    }
)


# ==========================================================
# PANEL SUPERIOR: MEDELLÍN
# ==========================================================

datos_medellin = (
    df_plot[
        df_plot["municipio"] == municipio_medellin
    ]
    .sort_values("anio")
)

ax1.plot(
    datos_medellin["anio"],
    datos_medellin["total_afectados"],
    marker="o",
    linewidth=2.2,
    color="#1f77b4",
    label=municipio_medellin
)

ax1.set_title(
    "Medellín",
    fontsize=12,
    pad=8
)

ax1.set_ylabel(
    "Personas afectadas",
    fontsize=11,
    labelpad=15
)

ax1.ticklabel_format(
    style="plain",
    axis="y"
)

ax1.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

ax1.set_ylim(
    0,
    datos_medellin["total_afectados"].max() * 1.08
)

ax1.set_xticks(range(2014, 2025))
ax1.tick_params(
    axis="x",
    labelbottom=True,
    labelsize=10
)

ax1.legend(
    title="Municipio",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    fontsize=9,
    title_fontsize=10,
    frameon=False
)


# ==========================================================
# PANEL INFERIOR: OTROS MUNICIPIOS
# ==========================================================

for municipio in municipios_otros:

    datos = (
        df_plot[
            df_plot["municipio"] == municipio
        ]
        .sort_values("anio")
    )

    ax2.plot(
        datos["anio"],
        datos["total_afectados"],
        marker="o",
        linewidth=2.2,
        label=municipio
    )

ax2.set_title(
    "Otros municipios del Top 10",
    fontsize=12,
    pad=8
)

ax2.set_xlabel(
    "Año",
    fontsize=11,
    labelpad=15
)

ax2.set_ylabel(
    "Personas afectadas",
    fontsize=11,
    labelpad=15
)

ax2.ticklabel_format(
    style="plain",
    axis="y"
)

ax2.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

max_otros = df_plot[
    df_plot["municipio"].isin(municipios_otros)
]["total_afectados"].max()

ax2.set_ylim(
    0,
    max_otros * 1.10
)

ax2.set_xticks(range(2014, 2025))
ax2.tick_params(
    axis="x",
    labelsize=10
)

ax2.legend(
    title="Municipio",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    fontsize=9,
    title_fontsize=10,
    frameon=False
)


# ==========================================================
# FORMATO GENERAL
# ==========================================================

fig.suptitle(
    "Evolución anual de las personas afectadas en los municipios "
    "del Top 10 de Antioquia, 2014–2024",
    fontsize=14,
    y=0.98
)

plt.tight_layout()

plt.subplots_adjust(
    top=0.90,
    right=0.82
)

plt.show()